# 08 — MLflow & Machine Learning in Fabric Spark

Fabric has MLflow built in (experiments/runs/model registry are workspace items, no setup
required). This notebook trains a fraud-detection style classifier on the transactions data from
notebook 07, tracks it with MLflow, registers it, then does batch scoring at Spark scale with a
`mlflow.pyfunc` UDF.


## 1. Feature engineering with Spark ML

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

df = spark.read.table("silver_transactions").withColumn(
    "is_flagged", (F.col("amount_usd") > 10000).cast("int")   # synthetic label for demo purposes
)

categorical_indexer = StringIndexer(
    inputCol="channel", outputCol="channel_idx", handleInvalid="keep"
)
assembler = VectorAssembler(
    inputCols=["amount_usd", "channel_idx"], outputCol="features", handleInvalid="skip"
)

prep_pipeline = Pipeline(stages=[categorical_indexer, assembler])
df_features = prep_pipeline.fit(df).transform(df)

train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)


## 2. Train and track with MLflow autologging

In [ ]:
import mlflow
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

mlflow.set_experiment("fraud-flagging-transactions")
mlflow.spark.autolog()   # auto-logs params, metrics, and the fitted model artifact

with mlflow.start_run(run_name="rf_baseline") as run:
    rf = RandomForestClassifier(
        featuresCol="features", labelCol="is_flagged", numTrees=100, maxDepth=6, seed=42
    )
    model = rf.fit(train_df)

    predictions = model.transform(test_df)
    evaluator = BinaryClassificationEvaluator(labelCol="is_flagged", metricName="areaUnderROC")
    auc = evaluator.evaluate(predictions)

    mlflow.log_metric("test_auc", auc)   # manual metric on top of autologging
    print(f"Run ID: {run.info.run_id}  Test AUC: {auc:.4f}")


## 3. Comparing runs / hyperparameter sweep

In [ ]:
for n_trees, depth in [(50, 4), (100, 6), (200, 8)]:
    with mlflow.start_run(run_name=f"rf_trees{n_trees}_depth{depth}"):
        rf = RandomForestClassifier(
            featuresCol="features", labelCol="is_flagged",
            numTrees=n_trees, maxDepth=depth, seed=42,
        )
        m = rf.fit(train_df)
        preds = m.transform(test_df)
        auc = evaluator.evaluate(preds)
        mlflow.log_params({"numTrees": n_trees, "maxDepth": depth})
        mlflow.log_metric("test_auc", auc)


## 4. Registering the best model

In [ ]:
best_run_id = run.info.run_id   # in practice: query mlflow.search_runs() and pick the best AUC

model_uri = f"runs:/{best_run_id}/model"
registered = mlflow.register_model(model_uri=model_uri, name="fraud_flagging_rf")
print(f"Registered as {registered.name} version {registered.version}")


## 5. Batch scoring at Spark scale with a `pyfunc` UDF

This is the pattern for scoring millions of rows without collecting data to a single driver.

In [ ]:
import mlflow.pyfunc

score_udf = mlflow.pyfunc.spark_udf(
    spark,
    model_uri=f"models:/fraud_flagging_rf/{registered.version}",
    result_type="double",
)

df_to_score = spark.read.table("silver_transactions").withColumn(
    "features_input", F.struct("amount_usd", "channel")  # adapt to your model's expected input
)

df_scored = df_to_score.withColumn("fraud_score", score_udf("features_input"))
df_scored.write.format("delta").mode("overwrite").saveAsTable("gold_transaction_fraud_scores")


## 6. Loading a registered model directly (outside Spark, e.g. for a single prediction)

In [ ]:
import mlflow.spark

loaded_model = mlflow.spark.load_model(f"models:/fraud_flagging_rf/{registered.version}")
sample = test_df.limit(5)
loaded_model.transform(sample).select("txn_id", "is_flagged", "prediction").show()


## 7. Where this fits in the bigger picture
- Notebook 07's Gold table is the **feature source** for this notebook
- The registered model version can be referenced from a **Fabric Data Pipeline** to run scheduled
  batch scoring the same way notebook 07 runs scheduled ETL
- Combine with notebook 05's tuning techniques when scoring at very large volumes

**Series complete.** You now have runnable, real-world coverage of: fundamentals, Delta Lake,
ingestion/transformation, Spark SQL + notebookutils, performance tuning, structured streaming,
an end-to-end medallion pipeline, and ML/MLflow — the full surface area of Fabric Spark.